In [2]:
from fastavro import parse_schema, schemaless_writer, schemaless_reader
import json
from dataclasses import asdict
from equity_ops.contracts import TradeEvent
from tests.fixtures import TRADE_EVENTS
import io
from confluent_kafka import Producer, KafkaException



In [3]:
with open("schemas/trade_event.avsc") as schema_file:
    schema = parse_schema(json.load(schema_file))

In [4]:
record = asdict(TRADE_EVENTS[0])

In [5]:
record

{'trade_id': 'T1',
 'version': 1,
 'symbol': 'ABC',
 'signed_quantity': 100,
 'price_in_pence': 1000,
 'cancelled': False}

In [6]:
stream = io.BytesIO()

In [7]:
schemaless_writer(stream, schema, record)

In [8]:
stream.seek(0)

0

In [9]:
stream.getvalue()

b'\x04T1\x02\x06ABC\xc8\x01\xd0\x0f\x00'

In [10]:
producer = Producer({"bootstrap.servers": "localhost:9092"})

In [11]:
def delivery_report(err="", msg=""):
    if err:
        raise KafkaException(err)
    print(f"Topic: {msg.topic()}, Partition: {msg.partition()}, Offset: {msg.offset()}")

In [46]:
stream.seek(0)
producer.produce(
    "equity_ops.trades.raw",
    key=TRADE_EVENTS[0].trade_id.encode("utf-8"),
    value=stream.getvalue(),
    on_delivery=delivery_report,
)
remaining = producer.flush(10.0)

Topic: equity_ops.trades.raw, Partition: 0, Offset: 1
